<a href="https://colab.research.google.com/github/saacosta-bit/Sinanzas-Agente/blob/main/notebooks/sinanzas_agente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3
from datetime import datetime
from zoneinfo import ZoneInfo
from typing import Optional
from pydantic import BaseModel, Field
from langchain_core.tools import tool

DB_PATH = "sinanzas.db"
CATEGORIAS = ["Alimentación", "Transporte", "Servicios",
              "Entretenimiento", "Salud", "Educación", "Otros"]

def hoy_chile() -> str:
    return datetime.now(ZoneInfo("America/Santiago")).date().isoformat()

def init_db():
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS gastos (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                monto REAL NOT NULL,
                categoria TEXT NOT NULL,
                descripcion TEXT,
                fecha TEXT NOT NULL
            )""")

init_db()

class GastoInput(BaseModel):
    monto: float = Field(gt=0, description="Monto del gasto en pesos chilenos")
    categoria: str = Field(description=f"Categoría, una de: {', '.join(CATEGORIAS)}")
    descripcion: str = Field(default="", description="Detalle breve del gasto")
    fecha: Optional[str] = Field(default=None,
        description="Fecha en formato YYYY-MM-DD. Si no se indica, se usa hoy")

@tool("registrar_gasto", args_schema=GastoInput)
def registrar_gasto(monto: float, categoria: str,
                    descripcion: str = "", fecha: Optional[str] = None) -> str:
    """Registra un gasto del usuario en la base de datos.
    Úsala cuando el usuario cuente que gastó dinero en algo."""
    if categoria not in CATEGORIAS:
        categoria = "Otros"
    try:
        fecha = datetime.strptime(fecha, "%Y-%m-%d").date().isoformat() if fecha else hoy_chile()
    except ValueError:
        return "Error: la fecha debe tener formato YYYY-MM-DD."
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute(
            "INSERT INTO gastos (monto, categoria, descripcion, fecha) VALUES (?, ?, ?, ?)",
            (monto, categoria, descripcion, fecha))
    return f"Gasto registrado: ${monto:,.0f} en {categoria} ({fecha})."

In [ ]:
print(registrar_gasto.invoke({"monto": 5000, "categoria": "Alimentación",
                              "descripcion": "Almuerzo"}))

with sqlite3.connect(DB_PATH) as conn:
    print(conn.execute("SELECT * FROM gastos").fetchall())

Gasto registrado: $5,000 en Alimentación (2026-10-18).
[(1, 5000.0, 'Alimentación', 'Almuerzo', '2026-10-19'), (2, 15000.0, 'Alimentación', 'Almuerzo', '2026-10-18'), (3, 5000.0, 'Alimentación', 'Almuerzo', '2026-10-18')]


In [ ]:
def init_tabla_ingresos():
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS ingresos (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                monto REAL NOT NULL,
                fuente TEXT,
                fecha TEXT NOT NULL
            )""")

init_tabla_ingresos()

class IngresoInput(BaseModel):
    monto: float = Field(gt=0, description="Monto del ingreso en pesos chilenos")
    fuente: str = Field(default="Sueldo", description="Origen del ingreso (sueldo, honorarios, venta, etc.)")
    fecha: Optional[str] = Field(default=None,
        description="Fecha en formato YYYY-MM-DD. Si no se indica, se usa hoy")

@tool("registrar_ingreso", args_schema=IngresoInput)
def registrar_ingreso(monto: float, fuente: str = "Sueldo",
                      fecha: Optional[str] = None) -> str:
    """Registra un ingreso de dinero del usuario.
    Úsala cuando el usuario cuente cuánto ganó o recibió."""
    try:
        fecha = datetime.strptime(fecha, "%Y-%m-%d").date().isoformat() if fecha else hoy_chile()
    except ValueError:
        return "Error: la fecha debe tener formato YYYY-MM-DD."
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute(
            "INSERT INTO ingresos (monto, fuente, fecha) VALUES (?, ?, ?)",
            (monto, fuente, fecha))
    return f"Ingreso registrado: ${monto:,.0f} por {fuente} ({fecha})."

In [ ]:
def init_tabla_metas():
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS metas (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                nombre TEXT NOT NULL,
                monto_objetivo REAL NOT NULL,
                plazo_meses INTEGER NOT NULL,
                fecha_inicio TEXT NOT NULL
            )""")

init_tabla_metas()

class MetaInput(BaseModel):
    nombre: str = Field(description="Nombre de la meta (ej. 'Notebook nuevo')")
    monto_objetivo: float = Field(gt=0, description="Monto total a ahorrar en pesos chilenos")
    plazo_meses: int = Field(gt=0, description="Plazo en meses para lograr la meta")

@tool("crear_meta_ahorro", args_schema=MetaInput)
def crear_meta_ahorro(nombre: str, monto_objetivo: float, plazo_meses: int) -> str:
    """Crea una meta de ahorro del usuario.
    Úsala cuando el usuario diga que quiere ahorrar una cantidad en un plazo."""
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute(
            "INSERT INTO metas (nombre, monto_objetivo, plazo_meses, fecha_inicio) VALUES (?, ?, ?, ?)",
            (nombre, monto_objetivo, plazo_meses, hoy_chile()))
    mensual = monto_objetivo / plazo_meses
    return (f"Meta creada: '{nombre}' por ${monto_objetivo:,.0f} en {plazo_meses} meses "
            f"(equivale a ${mensual:,.0f} por mes).")

In [ ]:
print(registrar_ingreso.invoke({"monto": 800000, "fuente": "Sueldo"}))
print(crear_meta_ahorro.invoke({"nombre": "Notebook nuevo",
                                "monto_objetivo": 600000, "plazo_meses": 6}))

with sqlite3.connect(DB_PATH) as conn:
    print(conn.execute("SELECT * FROM ingresos").fetchall())
    print(conn.execute("SELECT * FROM metas").fetchall())

Ingreso registrado: $800,000 por Sueldo (2026-10-18).
Meta creada: 'Notebook nuevo' por $600,000 en 6 meses (equivale a $100,000 por mes).
[(1, 800000.0, 'Sueldo', '2026-10-18')]
[(1, 'Notebook nuevo', 600000.0, 6, '2026-10-18')]


In [ ]:
class ConsultaInput(BaseModel):
    categoria: Optional[str] = Field(default=None,
        description=f"Categoría a filtrar (opcional): {', '.join(CATEGORIAS)}")
    mes: Optional[str] = Field(default=None,
        description="Mes a consultar en formato YYYY-MM (opcional). Si no se indica, trae todos")

@tool("consultar_gastos", args_schema=ConsultaInput)
def consultar_gastos(categoria: Optional[str] = None, mes: Optional[str] = None) -> str:
    """Consulta los gastos registrados, con filtro opcional por categoría y mes.
    Devuelve el detalle y el total. Úsala cuando el usuario pregunte cuánto gastó."""
    query = "SELECT fecha, categoria, descripcion, monto FROM gastos WHERE 1=1"
    params = []
    if categoria:
        query += " AND categoria = ?"
        params.append(categoria)
    if mes:
        query += " AND substr(fecha, 1, 7) = ?"
        params.append(mes)
    query += " ORDER BY fecha DESC"

    with sqlite3.connect(DB_PATH) as conn:
        filas = conn.execute(query, params).fetchall()

    if not filas:
        return "No hay gastos registrados con esos filtros."
    total = sum(f[3] for f in filas)
    detalle = "\n".join(f"- {f[0]} | {f[1]} | {f[2] or 'sin descripción'} | ${f[3]:,.0f}" for f in filas)
    return f"{detalle}\nTotal: ${total:,.0f} ({len(filas)} gastos)"

In [ ]:
!pip install -q faiss-cpu langchain-community langchain-text-splitters

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# CONTENIDO DE EJEMPLO: reemplázalo por guías reales y cita sus fuentes en el informe
GUIAS = [
    "Regla 50/30/20: destina 50% del ingreso a necesidades básicas, 30% a gustos personales y 20% a ahorro o pago de deudas.",
    "Gastos hormiga: son gastos pequeños y frecuentes, como cafés o snacks diarios, que sumados al mes pueden representar un monto importante. Anótalos para hacerlos visibles.",
    "Fondo de emergencia: apunta a reunir entre 3 y 6 meses de gastos básicos en una cuenta separada y de fácil acceso.",
    "Para salir de deudas, prioriza pagar primero la de mayor tasa de interés y mantén al día los pagos mínimos de las demás.",
    "Ahorra primero: transfiere un porcentaje fijo de tu ingreso a una cuenta de ahorro apenas lo recibes, en vez de ahorrar lo que sobre a fin de mes.",
    "Antes de una compra no planificada, espera 24 horas. Si sigues queriéndola, revisa si cabe en tu presupuesto del mes.",
]

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.create_documents(GUIAS)
vector_store = FAISS.from_documents(chunks, embeddings)

class GuiaInput(BaseModel):
    pregunta: str = Field(description="Consulta del usuario sobre finanzas personales")

@tool("buscar_guias_financieras", args_schema=GuiaInput)
def buscar_guias_financieras(pregunta: str) -> str:
    """Busca consejos en las guías de educación financiera.
    Úsala cuando el usuario pregunte cómo ahorrar, presupuestar o manejar deudas."""
    resultados = vector_store.similarity_search(pregunta, k=2)
    return "\n".join(f"- {doc.page_content}" for doc in resultados)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


/tmp/ipykernel_3430/1551477744.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
registrar_gasto.invoke({"monto": 2500, "categoria": "Alimentación", "descripcion": "Café"})
registrar_gasto.invoke({"monto": 1200, "categoria": "Transporte", "descripcion": "Metro"})

print(consultar_gastos.invoke({}))
print("---")
print(consultar_gastos.invoke({"categoria": "Alimentación", "mes": "2026-10"}))
print("---")
print(buscar_guias_financieras.invoke({"pregunta": "¿Cómo puedo ahorrar más dinero?"}))

- 2026-10-19 | Alimentación | Almuerzo | $5,000
- 2026-10-18 | Alimentación | Almuerzo | $15,000
- 2026-10-18 | Alimentación | Almuerzo | $5,000
- 2026-10-18 | Alimentación | Café | $2,500
- 2026-10-18 | Transporte | Metro | $1,200
Total: $28,700 (5 gastos)
---
- 2026-10-19 | Alimentación | Almuerzo | $5,000
- 2026-10-18 | Alimentación | Almuerzo | $15,000
- 2026-10-18 | Alimentación | Almuerzo | $5,000
- 2026-10-18 | Alimentación | Café | $2,500
Total: $27,500 (4 gastos)
---
- Ahorra primero: transfiere un porcentaje fijo de tu ingreso a una cuenta de ahorro apenas lo recibes, en vez de ahorrar lo que sobre a fin de mes.
- Regla 50/30/20: destina 50% del ingreso a necesidades básicas, 30% a gustos personales y 20% a ahorro o pago de deudas.


In [ ]:
class HormigaInput(BaseModel):
    mes: Optional[str] = Field(default=None,
        description="Mes en formato YYYY-MM. Si se omite, se usa el mes actual")
    umbral_monto: float = Field(default=5000, gt=0,
        description="Monto máximo (CLP) para considerar un gasto como pequeño")
    min_repeticiones: int = Field(default=3, ge=2,
        description="Veces mínimas que debe repetirse el gasto en el mes")

@tool("detectar_gastos_hormiga", args_schema=HormigaInput)
def detectar_gastos_hormiga(mes: Optional[str] = None, umbral_monto: float = 5000,
                            min_repeticiones: int = 3) -> str:
    """Detecta gastos hormiga: gastos pequeños que se repiten muchas veces en el mes.
    Úsala cuando el usuario quiera saber en qué se le va el dinero en cosas pequeñas."""
    mes = mes or hoy_chile()[:7]
    with sqlite3.connect(DB_PATH) as conn:
        total_mes = conn.execute(
            "SELECT COALESCE(SUM(monto), 0) FROM gastos WHERE substr(fecha, 1, 7) = ?",
            (mes,)).fetchone()[0]
        filas = conn.execute("""
            SELECT categoria, LOWER(TRIM(descripcion)), COUNT(*), SUM(monto)
            FROM gastos
            WHERE substr(fecha, 1, 7) = ? AND monto <= ?
            GROUP BY categoria, LOWER(TRIM(descripcion))
            HAVING COUNT(*) >= ?
            ORDER BY SUM(monto) DESC""", (mes, umbral_monto, min_repeticiones)).fetchall()

    if not filas:
        return f"No se detectaron gastos hormiga en {mes}."
    suma = sum(f[3] for f in filas)
    pct = (suma / total_mes * 100) if total_mes else 0
    detalle = "\n".join(
        f"- {f[1] or 'sin descripción'} ({f[0]}): {f[2]} veces, ${f[3]:,.0f}" for f in filas)
    return (f"Gastos hormiga en {mes}:\n{detalle}\n"
            f"Total: ${suma:,.0f}, equivale al {pct:.1f}% de tus gastos del mes.")

In [ ]:
import math

CATEGORIAS_FLEXIBLES = ["Entretenimiento", "Alimentación", "Otros"]

class PlanInput(BaseModel):
    nombre_meta: Optional[str] = Field(default=None,
        description="Nombre de la meta. Si se omite, se usa la más reciente")
    mes: Optional[str] = Field(default=None,
        description="Mes a analizar en formato YYYY-MM. Si se omite, el mes actual")

@tool("calcular_plan_ahorro", args_schema=PlanInput)
def calcular_plan_ahorro(nombre_meta: Optional[str] = None, mes: Optional[str] = None) -> str:
    """Calcula si el usuario puede cumplir una meta de ahorro con sus ingresos y gastos
    del mes, y sugiere recortes por categoría si no alcanza."""
    mes = mes or hoy_chile()[:7]
    with sqlite3.connect(DB_PATH) as conn:
        if nombre_meta:
            meta = conn.execute(
                "SELECT nombre, monto_objetivo, plazo_meses FROM metas "
                "WHERE LOWER(nombre) = LOWER(?) ORDER BY id DESC LIMIT 1", (nombre_meta,)).fetchone()
        else:
            meta = conn.execute(
                "SELECT nombre, monto_objetivo, plazo_meses FROM metas ORDER BY id DESC LIMIT 1").fetchone()
        ingresos = conn.execute(
            "SELECT COALESCE(SUM(monto), 0) FROM ingresos WHERE substr(fecha, 1, 7) = ?",
            (mes,)).fetchone()[0]
        por_cat = conn.execute(
            "SELECT categoria, SUM(monto) FROM gastos WHERE substr(fecha, 1, 7) = ? "
            "GROUP BY categoria ORDER BY SUM(monto) DESC", (mes,)).fetchall()

    if not meta:
        return "No hay metas de ahorro registradas. Crea una primero."
    if ingresos == 0:
        return f"No hay ingresos registrados en {mes}. Registra un ingreso para calcular el plan."

    nombre, objetivo, plazo = meta
    gastos = sum(m for _, m in por_cat)
    disponible = ingresos - gastos
    requerido = objetivo / plazo

    resumen = (f"Meta '{nombre}': ${objetivo:,.0f} en {plazo} meses (${requerido:,.0f} por mes). "
               f"Ingresos de {mes}: ${ingresos:,.0f}, gastos: ${gastos:,.0f}, "
               f"disponible: ${disponible:,.0f}.")
    if disponible >= requerido:
        return resumen + " La meta es alcanzable con tu ritmo actual."

    deficit = requerido - disponible
    flexibles = [(c, m) for c, m in por_cat if c in CATEGORIAS_FLEXIBLES]
    total_flex = sum(m for _, m in flexibles)
    sugerencias = []
    for c, m in flexibles:
        recorte = min(deficit * m / total_flex, m) if total_flex else 0
        sugerencias.append(f"- Reducir {c} en ${recorte:,.0f} (gastas ${m:,.0f})")
    texto = f"{resumen}\nTe faltan ${deficit:,.0f} al mes."
    if sugerencias:
        texto += "\nRecortes sugeridos:\n" + "\n".join(sugerencias)
    if total_flex < deficit:
        texto += "\nAun recortando todo lo flexible no alcanza."
    if disponible > 0:
        texto += f"\nAlternativa: ampliar el plazo a {math.ceil(objetivo / disponible)} meses."
    return texto

In [ ]:
for _ in range(5):
    registrar_gasto.invoke({"monto": 2500, "categoria": "Alimentación", "descripcion": "Café"})
registrar_gasto.invoke({"monto": 300000, "categoria": "Servicios", "descripcion": "Arriendo"})
registrar_gasto.invoke({"monto": 250000, "categoria": "Alimentación", "descripcion": "Supermercado"})
registrar_gasto.invoke({"monto": 180000, "categoria": "Entretenimiento", "descripcion": "Salidas"})

print(detectar_gastos_hormiga.invoke({}))
print("---")
print(calcular_plan_ahorro.invoke({}))

Gastos hormiga en 2026-10:
- café (Alimentación): 6 veces, $15,000
Total: $15,000, equivale al 1.9% de tus gastos del mes.
---
Meta 'Notebook nuevo': $600,000 en 6 meses ($100,000 por mes). Ingresos de 2026-10: $800,000, gastos: $771,200, disponible: $28,800.
Te faltan $71,200 al mes.
Recortes sugeridos:
- Reducir Alimentación en $43,932 (gastas $290,000)
- Reducir Entretenimiento en $27,268 (gastas $180,000)
Alternativa: ampliar el plazo a 21 meses.


In [ ]:
def mes_anterior(mes: str) -> str:
    anio, m = map(int, mes.split("-"))
    return f"{anio - 1}-12" if m == 1 else f"{anio}-{m - 1:02d}"

def gasto_por_categoria(mes: str) -> dict:
    with sqlite3.connect(DB_PATH) as conn:
        return dict(conn.execute(
            "SELECT categoria, SUM(monto) FROM gastos "
            "WHERE substr(fecha, 1, 7) = ? GROUP BY categoria", (mes,)).fetchall())

class AlertaInput(BaseModel):
    categoria: Optional[str] = Field(default=None,
        description="Categoría a revisar. Si se omite, se revisan todas")
    mes: Optional[str] = Field(default=None,
        description="Mes en formato YYYY-MM. Si se omite, el mes actual")
    umbral_porcentaje: float = Field(default=30, gt=0,
        description="Aumento porcentual respecto al mes anterior que dispara la alerta")
    limite_monto: Optional[float] = Field(default=None, gt=0,
        description="Límite de gasto en pesos definido por el usuario (opcional)")

@tool("generar_alerta_categoria", args_schema=AlertaInput)
def generar_alerta_categoria(categoria: Optional[str] = None, mes: Optional[str] = None,
                             umbral_porcentaje: float = 30,
                             limite_monto: Optional[float] = None) -> str:
    """Genera alertas cuando el gasto de una categoría supera el límite del usuario
    o aumenta mucho respecto al mes anterior."""
    mes = mes or hoy_chile()[:7]
    prev = mes_anterior(mes)
    actual = gasto_por_categoria(mes)
    anterior = gasto_por_categoria(prev)
    categorias = [categoria] if categoria else list(actual.keys())

    alertas = []
    for c in categorias:
        g, p = actual.get(c, 0), anterior.get(c, 0)
        if p > 0 and g > p * (1 + umbral_porcentaje / 100):
            alertas.append(f"ALERTA en {c}: ${g:,.0f} en {mes}, "
                           f"{(g - p) / p * 100:.0f}% más que en {prev} (${p:,.0f}).")
        if limite_monto and g > limite_monto:
            alertas.append(f"ALERTA en {c}: superaste tu límite de ${limite_monto:,.0f} "
                           f"(llevas ${g:,.0f}).")
    if not alertas:
        return f"Sin alertas en {mes}: tus gastos están dentro de lo esperado."
    return "\n".join(alertas)

In [ ]:
class ReporteInput(BaseModel):
    mes: Optional[str] = Field(default=None,
        description="Mes en formato YYYY-MM. Si se omite, el mes actual")

@tool("generar_reporte_mensual", args_schema=ReporteInput)
def generar_reporte_mensual(mes: Optional[str] = None) -> str:
    """Genera el resumen del mes: ingresos, gastos por categoría, ahorro
    y comparación con el mes anterior."""
    mes = mes or hoy_chile()[:7]
    prev = mes_anterior(mes)
    with sqlite3.connect(DB_PATH) as conn:
        ingresos = conn.execute(
            "SELECT COALESCE(SUM(monto), 0) FROM ingresos WHERE substr(fecha, 1, 7) = ?",
            (mes,)).fetchone()[0]
    por_cat = gasto_por_categoria(mes)
    gastos = sum(por_cat.values())
    gastos_prev = sum(gasto_por_categoria(prev).values())
    ahorro = ingresos - gastos

    lineas = [f"REPORTE {mes}",
              f"Ingresos: ${ingresos:,.0f}",
              f"Gastos: ${gastos:,.0f}",
              f"Ahorro: ${ahorro:,.0f}" +
              (f" ({ahorro / ingresos * 100:.1f}% del ingreso)" if ingresos else ""),
              "Gastos por categoría:"]
    for c, m in sorted(por_cat.items(), key=lambda x: -x[1]):
        lineas.append(f"- {c}: ${m:,.0f} ({m / gastos * 100:.1f}%)")
    if gastos_prev > 0:
        dif = (gastos - gastos_prev) / gastos_prev * 100
        lineas.append(f"Comparación con {prev}: gastos {dif:+.1f}% (${gastos_prev:,.0f} el mes anterior).")
    else:
        lineas.append(f"Sin datos de {prev} para comparar.")
    return "\n".join(lineas)

In [ ]:
# Datos simulados del mes anterior
registrar_ingreso.invoke({"monto": 800000, "fuente": "Sueldo", "fecha": "2026-09-05"})
registrar_gasto.invoke({"monto": 300000, "categoria": "Servicios", "descripcion": "Arriendo", "fecha": "2026-09-05"})
registrar_gasto.invoke({"monto": 200000, "categoria": "Alimentación", "descripcion": "Supermercado", "fecha": "2026-09-10"})
registrar_gasto.invoke({"monto": 80000, "categoria": "Entretenimiento", "descripcion": "Salidas", "fecha": "2026-09-20"})

print(generar_alerta_categoria.invoke({}))
print("---")
print(generar_reporte_mensual.invoke({}))

ALERTA en Alimentación: $290,000 en 2026-10, 45% más que en 2026-09 ($200,000).
ALERTA en Entretenimiento: $180,000 en 2026-10, 125% más que en 2026-09 ($80,000).
---
REPORTE 2026-10
Ingresos: $800,000
Gastos: $771,200
Ahorro: $28,800 (3.6% del ingreso)
Gastos por categoría:
- Servicios: $300,000 (38.9%)
- Alimentación: $290,000 (37.6%)
- Entretenimiento: $180,000 (23.3%)
- Transporte: $1,200 (0.2%)
Comparación con 2026-09: gastos +33.0% ($580,000 el mes anterior).
